In [1]:
import joblib
import shap
import pandas as pd
import numpy as np
from sklearn import metrics
from sklearn.metrics import r2_score
from sklearn import pipeline
import os
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

## Prediction by field with trained per point

In [2]:
KNN_FILENAME = "./models/knn.sav"

In [3]:
SCALER_KNN_FILE = "./models/scaler_knn.pkl"

In [4]:
KNN_Model = joblib.load(KNN_FILENAME)

In [5]:
SCALER_KNN = joblib.load(SCALER_KNN_FILE)

In [6]:
yield_df = pd.read_csv("./data_1/all_yields_.csv")

In [7]:
feature_names = ["Moisture", "Elevation" , "xcoord" ,
                 "ycoord", "area", "Organic M", "Ca", "Mg", "Mn", "B",
                 "Cu", "Mo", "Fe", "Zn", "S", "P", "K", "Na", "pH", "C.E.C",
                 "year", "Crop_22", "Crop_34", "Crop_41", "Crop_45", "Crop_174", 

    "mean_tempmax_month_5", "mean_tempmax_month_6", "mean_tempmax_month_7",
    "mean_tempmax_month_8", "mean_tempmax_month_9", "mean_tempmax_month_10",
    "std_tempmax_month_5", "std_tempmax_month_6", "std_tempmax_month_7",
    "std_tempmax_month_8", "std_tempmax_month_9", "std_tempmax_month_10",

    "mean_tempmin_month_5", "mean_tempmin_month_6", "mean_tempmin_month_7",
    "mean_tempmin_month_8", "mean_tempmin_month_9", "mean_tempmin_month_10",
    "std_tempmin_month_5", "std_tempmin_month_6", "std_tempmin_month_7",
    "std_tempmin_month_8", "std_tempmin_month_9", "std_tempmin_month_10",

    "mean_temp_month_5", "mean_temp_month_6", "mean_temp_month_7",
    "mean_temp_month_8", "mean_temp_month_9", "mean_temp_month_10",
    "std_temp_month_5", "std_temp_month_6", "std_temp_month_7",
    "std_temp_month_8", "std_temp_month_9", "std_temp_month_10",
    
    "mean_humidity_month_5", "mean_humidity_month_6", "mean_humidity_month_7",
    "mean_humidity_month_8", "mean_humidity_month_9", "mean_humidity_month_10",
    "std_humidity_month_5", "std_humidity_month_6", "std_humidity_month_7",
    "std_humidity_month_8", "std_humidity_month_9", "std_humidity_month_10",
    
    "mean_precip_month_5", "mean_precip_month_6", "mean_precip_month_7",
    "mean_precip_month_8", "mean_precip_month_9", "mean_precip_month_10",
    "std_precip_month_5", "std_precip_month_6", "std_precip_month_7",
    "std_precip_month_8", "std_precip_month_9", "std_precip_month_10",
    
    "mean_cloudcover_month_5", "mean_cloudcover_month_6", "mean_cloudcover_month_7",
    "mean_cloudcover_month_8", "mean_cloudcover_month_9", "mean_cloudcover_month_10",
    "std_cloudcover_month_5", "std_cloudcover_month_6", "std_cloudcover_month_7",
    "std_cloudcover_month_8", "std_cloudcover_month_9", "std_cloudcover_month_10",
    ]

In [8]:
len(feature_names)

98

In [9]:
FEATURE_NUMBER = dict()

for i in range(len(feature_names)):
    FEATURE_NUMBER[i] = feature_names[i]

In [10]:
# FEATURE_NUMBER

In [11]:
len(yield_df['area'].unique())

14

In [12]:
agg_yield = yield_df.groupby(['area']).mean().reset_index()

In [13]:
agg_yield

,area,VRYIELDMAS,SECTIONID,Moisture,Elevation,xcoord,ycoord,Organic M,Ca,Mg,...,mean_cloudcover_month_7,mean_cloudcover_month_8,mean_cloudcover_month_9,mean_cloudcover_month_10,std_cloudcover_month_5,std_cloudcover_month_6,std_cloudcover_month_7,std_cloudcover_month_8,std_cloudcover_month_9,std_cloudcover_month_10
0,4.710000,6.522435,-166074.972071,14.438486,264.534446,25.841440,48.828679,6.000000,4779.00000,168.000000,...,59.546591,50.619779,58.494089,60.158872,24.934243,23.353861,22.851911,24.544182,28.456974,27.299803
1,4.880000,6.403571,-268817.368052,14.791212,255.211345,25.843200,48.833150,6.300000,4984.00000,166.000000,...,59.455850,50.699895,58.107462,60.843441,24.857183,23.064423,22.911247,24.309757,28.508561,26.783939
2,4.910000,6.777394,-305216.599688,15.474193,253.375394,25.843553,48.831344,7.100000,3992.00000,154.000000,...,58.838145,51.078787,58.150753,62.525111,24.713859,22.747527,23.252069,24.165945,28.604143,25.753850
3,4.920000,7.137584,-145456.966752,14.175549,260.828959,25.846213,48.830013,6.200000,4700.00000,153.000000,...,59.298495,49.986805,56.877862,63.553392,24.639067,22.646493,22.781967,24.100500,29.066325,25.558244
4,4.930000,6.869611,-170262.758586,14.960648,257.508980,25.844946,48.830677,5.500000,4881.00000,135.000000,...,59.879788,50.699504,58.737071,58.915418,25.117116,23.698882,22.766600,24.617829,28.199252,27.823523
5,4.950000,6.547655,-288433.259634,15.168573,257.505742,25.845190,48.833950,5.600000,4983.00000,136.000000,...,59.452000,50.702149,57.895169,61.252126,24.848927,22.998483,22.921758,24.206405,28.515469,26.472055
6,5.134886,5.940192,-301790.130194,17.521878,227.356251,25.841610,48.834690,6.307091,4562.60694,144.380222,...,56.012590,58.667304,68.969252,54.071677,24.202474,22.386957,25.835063,24.847684,24.932413,28.943302
7,5.140000,6.902517,-272131.682813,14.887589,256.887134,25.847208,48.834740,5.200000,4629.00000,141.000000,...,59.496053,50.699503,58.014209,61.233000,24.825901,23.020243,22.882701,24.231076,28.459313,26.510427
8,5.170000,5.527148,-277751.035507,15.437120,254.866854,25.838902,48.829177,7.000000,3890.00000,169.000000,...,58.747900,51.032979,58.400687,61.908156,24.766231,22.881774,23.292288,24.369050,28.705874,26.262671
9,5.180000,4.754063,-271491.447950,15.611505,252.865391,25.841477,48.835082,9.500000,4901.00000,100.000000,...,59.009689,50.887336,58.549735,59.872533,25.019271,23.281554,23.221397,24.626580,28.737513,27.460445


In [14]:
weather = pd.read_csv('./data_1/weather_data/wether_.csv')
soil_analysis = pd.read_csv('./data_1/field_62/soil_analysis_field_62.csv')

In [15]:
len(soil_analysis)

13

In [16]:
# weather

In [17]:
# soil_analysis

In [18]:
clear_soil_analysis = soil_analysis.drop(columns=['Name', 'CROPNAME_', 'P_Index', 'K_Index', 'Mg_Index'])

In [19]:
clear_soil_analysis.columns

Index(['area', 'Organic M', 'Ca', 'Mg', 'Mn', 'B', 'Cu', 'Mo', 'Fe', 'Zn', 'S',
       'P', 'K', 'Na', 'pH', 'C.E.C'],
      dtype='object')

In [20]:
weather_2022 = weather[weather["year"] == 2022]

In [21]:
df = pd.merge(clear_soil_analysis, weather_2022, how='cross')

In [22]:
df.columns

Index(['area', 'Organic M', 'Ca', 'Mg', 'Mn', 'B', 'Cu', 'Mo', 'Fe', 'Zn',
       ...
       'mean_cloudcover_month_7', 'mean_cloudcover_month_8',
       'mean_cloudcover_month_9', 'mean_cloudcover_month_10',
       'std_cloudcover_month_5', 'std_cloudcover_month_6',
       'std_cloudcover_month_7', 'std_cloudcover_month_8',
       'std_cloudcover_month_9', 'std_cloudcover_month_10'],
      dtype='object', length=161)

In [23]:
len(df)

13

In [24]:
spatial_per_area = yield_df.groupby('area')[['Moisture', 'Elevation', 'xcoord', 'ycoord']].mean()
for col in ['Moisture', 'Elevation', 'xcoord', 'ycoord']:
    df[col] = df['area'].map(spatial_per_area[col])

df['Crop_22']  = 0
df['Crop_34']  = 1
df['Crop_41']  = 0
df['Crop_45']  = 0
df['Crop_174'] = 0

df['year'] = 2022

In [25]:
df.columns

Index(['area', 'Organic M', 'Ca', 'Mg', 'Mn', 'B', 'Cu', 'Mo', 'Fe', 'Zn',
       ...
       'std_cloudcover_month_10', 'Moisture', 'Elevation', 'xcoord', 'ycoord',
       'Crop_22', 'Crop_34', 'Crop_41', 'Crop_45', 'Crop_174'],
      dtype='object', length=170)

In [26]:
df

,area,Organic M,Ca,Mg,Mn,B,Cu,Mo,Fe,Zn,...,std_cloudcover_month_10,Moisture,Elevation,xcoord,ycoord,Crop_22,Crop_34,Crop_41,Crop_45,Crop_174
0,6.03,7.1,4207,103,132,2.33,4.5,0.03,68,1.7,...,24.506224,14.616713,233.958857,25.839844,48.830944,0,1,0,0,0
1,5.18,9.5,4901,100,146,5.28,4.8,0.01,35,2.9,...,24.506224,15.611505,252.865391,25.841477,48.835082,0,1,0,0,0
2,4.92,6.2,4700,153,128,2.39,5.3,0.02,186,1.4,...,24.506224,14.175549,260.828959,25.846213,48.830013,0,1,0,0,0
3,5.14,5.2,4629,141,165,2.54,5.3,0.03,132,1.5,...,24.506224,14.887589,256.887134,25.847208,48.834740,0,1,0,0,0
4,5.22,7.2,4628,118,231,4.28,5.1,0.02,98,1.8,...,24.506224,15.553322,252.990607,25.843439,48.836421,0,1,0,0,0
5,4.95,5.6,4983,136,196,2.96,5.1,0.02,127,1.4,...,24.506224,15.168573,257.505742,25.845190,48.833950,0,1,0,0,0
6,4.88,6.3,4984,166,201,3.86,5.6,0.02,116,1.7,...,24.506224,14.791212,255.211345,25.843200,48.833150,0,1,0,0,0
7,4.91,7.1,3992,154,140,2.56,6.0,0.02,150,1.7,...,24.506224,15.474193,253.375394,25.843553,48.831344,0,1,0,0,0
8,4.93,5.5,4881,135,135,2.15,5.1,0.02,155,1.3,...,24.506224,14.960648,257.508980,25.844946,48.830677,0,1,0,0,0
9,5.56,5.0,4144,158,141,2.04,5.5,0.02,193,1.4,...,24.506224,14.328061,257.014131,25.850607,48.832207,0,1,0,0,0


In [27]:
# df_encoded = pd.get_dummies(df)
# df_encoded = df_encoded.reindex(columns=feature_names, fill_value=0)

# X_ = df_encoded.iloc[[3]].astype(float)

In [28]:
X_ = df[feature_names].iloc[[3]].astype(float)
y = agg_yield["VRYIELDMAS"][3]

X_scaled = SCALER_KNN.transform(X_)
X_scaled = pd.DataFrame(X_scaled, columns=feature_names)

In [29]:
os.environ["LOKY_MAX_CPU_COUNT"] = "1"
y_pred = KNN_Model.predict(X_scaled.values)
error = abs(y - y_pred)

print("Prediction:", y_pred)
print("True value:", y)
print("Absolute error:", error)

Prediction: [3.55606282]
True value: 7.13758350996551
Absolute error: [3.58152069]


In [30]:
len(df)

13

In [31]:
os.environ["LOKY_MAX_CPU_COUNT"] = "1"

for i in range(13):
    X_ = df[feature_names].iloc[[3]].astype(float)
    y = agg_yield["VRYIELDMAS"][i]

    X_scaled = SCALER_KNN.transform(X_)
    X_scaled = pd.DataFrame(X_scaled, columns=feature_names)

    y_pred = KNN_Model.predict(X_scaled)
    error = abs(y - y_pred)

    print("=============")
    print("Prediction:", y_pred)
    print("True value:", y)
    print("Absolute error:", error)
    print("=============")
    print("               ")

Prediction: [3.55606282]
True value: 6.522435034489545
Absolute error: [2.96637222]
               
Prediction: [3.55606282]
True value: 6.403570970403429
Absolute error: [2.84750815]
               
Prediction: [3.55606282]
True value: 6.777394033719213
Absolute error: [3.22133122]
               
Prediction: [3.55606282]
True value: 7.13758350996551
Absolute error: [3.58152069]
               
Prediction: [3.55606282]
True value: 6.869611103194083
Absolute error: [3.31354829]
               


c:\Users\katja\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but KNeighborsRegressor was fitted without feature names
  warnings.warn(
c:\Users\katja\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but KNeighborsRegressor was fitted without feature names
  warnings.warn(
c:\Users\katja\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but KNeighborsRegressor was fitted without feature names
  warnings.warn(
c:\Users\katja\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but KNeighborsRegressor was fitted without feature names
  warnings.warn(
c:\Users\katja\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names,

Prediction: [3.55606282]
True value: 6.547654500799426
Absolute error: [2.99159168]
               
Prediction: [3.55606282]
True value: 5.940191675290858
Absolute error: [2.38412886]
               
Prediction: [3.55606282]
True value: 6.902516887299673
Absolute error: [3.34645407]
               
Prediction: [3.55606282]
True value: 5.527148096108187
Absolute error: [1.97108528]
               
Prediction: [3.55606282]
True value: 4.754063339560068
Absolute error: [1.19800052]
               
Prediction: [3.55606282]
True value: 6.154448603333566
Absolute error: [2.59838579]
               
Prediction: [3.55606282]
True value: 7.079657759793276
Absolute error: [3.52359494]
               
Prediction: [3.55606282]
True value: 7.052272238424906
Absolute error: [3.49620942]
               


c:\Users\katja\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but KNeighborsRegressor was fitted without feature names
  warnings.warn(
c:\Users\katja\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but KNeighborsRegressor was fitted without feature names
  warnings.warn(
c:\Users\katja\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but KNeighborsRegressor was fitted without feature names
  warnings.warn(
c:\Users\katja\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but KNeighborsRegressor was fitted without feature names
  warnings.warn(
c:\Users\katja\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names,